In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error
from viz_config import VizConfig

# ==========================================
# 0. Configuration and initialisation
# ==========================================
# Import the shared plotting style (viz_config.py) to keep validation figures consistent
VizConfig.set_style()

OUTPUT_DIR = "Stage2_Hypothesis_Verification"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Concentration scaling factor
# Scale ~1e-7 concentration values to ~1.0 to avoid numerical instability during nonlinear fitting
SCALE = 1e7

# ==========================================
# 1. Prepare the data
# ==========================================
print("Reading data...")
if not os.path.exists('data/train_dataset_ready.csv'):
    print("ERROR: train_dataset_ready.csv not found")
else:
    df = pd.read_csv('data/train_dataset_ready.csv')

    # Apply scaling to the concentration
    df['C_in_scaled'] = df['C_in'] * SCALE
    df['C_out_scaled'] = df['C_out'] * SCALE

    # Extract features for the fit
    V_in = df['V_in'].values
    Area = df['Area'].values
    Dist = df['Distance'].values
    C_in_scaled = df['C_in_scaled'].values
    C_out_scaled = df['C_out_scaled'].values

    # ==========================================
    # 2. Hypothesis formula definition
    # ==========================================
    # This structure was refined from the Stage 1 PySR results plus physical intuition
    # Form: C = C_in / [ (a * x) / (S - b * sqrt(x/v) + c) + d ]
    def hypothesis_formula(X, a, b, c, d):
        v, area, dist, c_in = X
        
        # Dimensional term: dist / v has units of time
        # 1e-6 Prevent division by zero
        time_scale = np.sqrt(dist / (v + 1e-6))
        
        # Effective-area term
        # Core denominator term - effective area of influence after diffusion
        # np.maximum Keep this term non-negative to preserve physical meaning
        effective_term = area - b * time_scale + c
        
        # Build the full denominator
        denominator = (a * dist) / (np.maximum(effective_term, 0.1)) + d
        
        return c_in / denominator

    # ==========================================
    # 3. Parameter fitting
    # ==========================================
    print("Running non-linear least-squares fit...")
    
    # Initial guess
    # Seeded from the Stage 1 symbolic-regression results
    # a(alpha)≈0.08, b(beta)≈0.6, c(gamma)≈26, d(delta)≈1.0
    p0 = [0.08, 0.6, 26.0, 1.0] 
    X_data = (V_in, Area, Dist, C_in_scaled)

    try:
        # Fit with scipy.optimize.curve_fit
        # bounds Bound the parameters to physically reasonable signs (e.g. all positive)
        popt, pcov = curve_fit(
            hypothesis_formula, 
            X_data, 
            C_out_scaled, 
            p0=p0, 
            maxfev=20000,
            bounds=([0, 0, 0, 0.5], [2.0, 10.0, 100.0, 2.0])
        )
        
        a_opt, b_opt, c_opt, d_opt = popt
        
        # Compute predictions with the optimised parameters
        y_pred = hypothesis_formula(X_data, *popt)
        
        print("\n" + "="*40)
        print("Fit succeeded - best parameters:")
        print(f"alpha (a) = {a_opt:.6f}")
        print(f"beta  (b) = {b_opt:.6f}")
        print(f"gamma (c) = {c_opt:.6f}")
        print(f"delta (d) = {d_opt:.6f}")

        # Compute metrics (R^2, RMSE)
        r2 = r2_score(C_out_scaled, y_pred)
        rmse = np.sqrt(mean_squared_error(C_out_scaled, y_pred))
        
        # Save final parameters to a text file
        with open(os.path.join(OUTPUT_DIR, "final_parameters.txt"), "w") as f:
            f.write(f"R2 Score: {r2:.6f}\nRMSE: {rmse:.6f}\n")
            f.write(f"alpha: {a_opt}\nbeta: {b_opt}\ngamma: {c_opt}\ndelta: {d_opt}\n")

        # ==========================================
        # 4. Visual validation 1: decay curves
        # ==========================================
        print("Plotting 1.pdf (decay curves)...")
        unique_cases = df['Case'].unique()
        np.random.seed(42) 
        # Draw 4 cases for display
        sample_cases = np.random.choice(unique_cases, 4, replace=False)
        
        # Local font sizes (feel free to pull from VizConfig instead)
        CASE_TITLE_SIZE = VizConfig.TITLE_SIZE
        CASE_LABEL_SIZE = VizConfig.LABEL_SIZE
        CASE_TICK_SIZE = VizConfig.TICK_SIZE
        CASE_LEGEND_SIZE = VizConfig.LEGEND_SIZE
        
        fig1 = plt.figure(figsize=(15, 10))
        unit_c = r"($10^{-7} \cdot \text{kg/m}^3$)"
        
        for i, case_id in enumerate(sample_cases):
            # Extract the current case data
            case_data = df[df['Case'] == case_id].sort_values('Distance')
            dist_case = case_data['Distance'].values
            c_true_case = case_data['C_out_scaled'].values
            v_case = case_data['V_in'].values
            a_case = case_data['Area'].values
            c_in_case = case_data['C_in_scaled'].values
            
            # Generate the predicted curve from the fitted parameters
            pred_case = hypothesis_formula((v_case, a_case, dist_case, c_in_case), *popt)
            
            ax = plt.subplot(2, 2, i+1)
            # Scatter the ground truth (CFD data)
            ax.plot(dist_case, c_true_case, 'o', color=VizConfig.COLOR_AXIS, markersize=4, 
                    alpha=0.6, label='CFD Data', rasterized=True)
            # Plot the predicted curve (proposed formula) - red highlight
            ax.plot(dist_case, pred_case, color=VizConfig.COLOR_HIGHLIGHT, linewidth=2, label='Proposed Formula')
            
            ax.set_title(f"Case: {case_id} (V={v_case[0]:.2f}m/s, Area={a_case[0]:.1f}$m^2$)", 
                         fontsize=CASE_TITLE_SIZE)
            ax.set_xlabel("Distance (m)", fontsize=CASE_LABEL_SIZE)
            ax.set_ylabel(f"Concentration {unit_c}", fontsize=CASE_LABEL_SIZE)
            
            ax.tick_params(axis='both', labelsize=CASE_TICK_SIZE)
            ax.legend(fontsize=CASE_LEGEND_SIZE)
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "1.pdf"), dpi=VizConfig.DPI)
        
        # ==========================================
        # 5. Visual validation 2: parity plot
        # ==========================================
        print("Plotting 2.pdf (parity plot)...")
        
        plt.figure(figsize=(8, 8))
        # Scatter predictions vs ground truth for every sample
        plt.plot(C_out_scaled, y_pred, 'o', color=VizConfig.COLOR_MAIN, markersize=2, 
                 alpha=0.1, rasterized=True)
        
        # Draw the 1:1 reference line (perfect prediction)
        plt.plot([C_out_scaled.min(), C_out_scaled.max()], 
                 [C_out_scaled.min(), C_out_scaled.max()], 
                 color=VizConfig.COLOR_HIGHLIGHT, linestyle='--', linewidth=2)
        
        plt.xlabel(f"True Concentration {unit_c}", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
        plt.ylabel(f"Predicted Concentration {unit_c}", fontsize=VizConfig.LABEL_SIZE, labelpad=10)
        plt.title(f"Parity Plot ($R^2={r2:.4f}$)", fontsize=VizConfig.TITLE_SIZE, pad=20)
        
        plt.xticks(fontsize=VizConfig.TICK_SIZE)
        plt.yticks(fontsize=VizConfig.TICK_SIZE)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "2.pdf"), dpi=VizConfig.DPI)

        print(f"\nDone!")
        print(f"Fit results saved successfully.")

    except Exception as e:
        print(f"Error during fitting: {e}")
        import traceback
        traceback.print_exc()